In [1]:
import os
from dotenv import load_dotenv

print("Exists:", os.path.exists("../openai.env"))

load_dotenv("../openai.env", override=True)
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Exists: True
Key loaded: True


In [2]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

/tmp/ipykernel_17/3401697195.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader


In [3]:
load_dotenv("../openai.env")

True

# 1. Loading Documents


- This is the python download script for wikipedia
```python

import requests

topics = ["Photosynthesis", "Water cycle", "Cellular respiration",
          "Plate tectonics", "Food chain"]

out_dir = "../resources"          # notebooks/ -> SemProject_MLEngine/resources
os.makedirs(out_dir, exist_ok=True)

headers = {"User-Agent": "SemProject-MLEngine/1.0 (student project)"}

for title in topics:
    r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1,
                "titles": title, "redirects": 1, "format": "json"},
        headers=headers, timeout=30,
    )
    page = next(iter(r.json()["query"]["pages"].values()))
    words = page["extract"].split(" ")
    text = " ".join(words[:1500])

    fname = f"{out_dir}/{title.lower().replace(' ', '_')}.txt"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(f"Source: Wikipedia, '{title}' (CC BY-SA)\n\n{text}")
    print(f"Saved {fname} ({len(text.split())} words)")

```


In [4]:
docs_path ="../../RAG_CONTEXT_DOCUMENTS"
loader= DirectoryLoader(docs_path, glob="*.pdf", loader_cls=PyPDFLoader, show_progress=True,)

documents=loader.load()
print(f"Loaded {len(documents)} documents")

 12%|█▎        | 1/8 [00:01<00:10,  1.51s/it]fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AEYEUS+MyriadPro-BoldCond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 86, '/FontDescriptor': IndirectObject(1574, 0, 134740929479216), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [452, 0, 0, 402, 0, 0, 0, 0, 0, 0, 0, 0, 0, 323, 0, 419, 0, 0, 0, 223, 0, 0, 0, 0, 0, 431, 0, 0, 294, 0, 0, 452]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AEYEUS+MyriadPro-BoldCond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 86, '/FontDescriptor': IndirectObject(1574, 0, 134740929479216), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [452, 0, 0, 402, 0, 0, 0, 0, 0, 0, 0, 0, 0, 323, 0, 419, 0, 0, 0, 223, 0, 0, 0, 0, 0, 431, 0, 0, 294, 0, 0, 452]}, but is no

Loaded 758 documents


- There are 758 total pages loaded, each page in pdf registers as a document

## 2. Splitting Documents into Chunks

### a) Fixed Chunking

In [5]:
fixed_splitter= CharacterTextSplitter(separator="",chunk_size=1000, chunk_overlap=100)
fixed_chunks= fixed_splitter.split_documents(documents)

chunk_lengths= []
for chunk in fixed_chunks:
    chunklength=len(chunk.page_content)
    chunk_lengths.append(chunklength)
print(chunk_lengths[:50])
print (f"Created {len(chunk_lengths)} chunks")

[209, 953, 1000, 1000, 1000, 1000, 1000, 482, 1000, 999, 702, 1000, 1000, 819, 999, 1000, 1000, 571, 1000, 1000, 1000, 463, 1000, 999, 730, 1000, 998, 948, 999, 1000, 999, 585, 1000, 998, 999, 267, 1000, 1000, 1000, 411, 1000, 999, 1000, 628, 1000, 794, 143, 39, 78, 513]
Created 1652 chunks


- We observe that chunks are many with fixed splitting

In [6]:
for i, chunk in enumerate(fixed_chunks[:10]):
    print(f"----Chunk {i+1}----")
    print(chunk.page_content)
    print()

----Chunk 1----
LAWS OF KENYA
THE ELECTION OFFENCES ACT
CAP. 66
Revised and published by the National Council for Law Reporting
with the authority of the Attorney-General as gazetted by the Government Printer
www.kenyalaw.org

----Chunk 2----
Kenya
Election Offences Act
Cap. 66
Legislation as at 31 December 2022
 
 
 
 
 
 
 
 
 
 
 
 
By Kenya Law and Laws.Africa. Share widely and freely.
www.kenyalaw.org | info@kenyalaw.org
FRBR URI: /akn/ke/act/2016/37/eng@2022-12-31
There is no copyright on the legislative content of this document.
This PDF copy is licensed under a Creative Commons Attribution NonCommercial ShareAlike 4.0 License (CC BY-NC-
SA 4.0). This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format
for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build
upon the material, you must license the modiﬁed material under identical terms. CC BY-NC-SA includes the fol

- The general observation is useless parts get chunked, such as the table of contents in chunk 3

### b) Recursive Splitting

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
recursive_splitter=RecursiveCharacterTextSplitter(separators=["\n\n","\n"," ",""], chunk_size=500, chunk_overlap=50)

In [8]:
recursive_chunks=recursive_splitter.split_documents(documents)

In [9]:
chunksizes=[]
for recursive_chunk in recursive_chunks:
    chunksizes.append(len(recursive_chunk.page_content))

print(chunksizes[:10])
print(len(chunksizes))
for i, recursive_chunk in enumerate(recursive_chunks[:10]):
    print(f"---Chunk {i+1}----")
    print(recursive_chunk.page_content)
    print()

[209, 409, 463, 79, 478, 373, 378, 411, 433, 378]
3012
---Chunk 1----
LAWS OF KENYA
THE ELECTION OFFENCES ACT
CAP. 66
Revised and published by the National Council for Law Reporting
with the authority of the Attorney-General as gazetted by the Government Printer
www.kenyalaw.org

---Chunk 2----
Kenya
Election Offences Act
Cap. 66
Legislation as at 31 December 2022
 
 
 
 
 
 
 
 
 
 
 
 
By Kenya Law and Laws.Africa. Share widely and freely.
www.kenyalaw.org | info@kenyalaw.org
FRBR URI: /akn/ke/act/2016/37/eng@2022-12-31
There is no copyright on the legislative content of this document.
This PDF copy is licensed under a Creative Commons Attribution NonCommercial ShareAlike 4.0 License (CC BY-NC-

---Chunk 3----
SA 4.0). This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format
for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build
upon the material, you must license the

- Recursive chunks are much more compared to fixed chunks
  

### c) Semantic Splitting

In [11]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter=SemanticChunker(OpenAIEmbeddings())

semantic_chunks=semantic_splitter.split_documents(documents)
semantic_chunk_lengths=[]
for chunk in semantic_chunks:
    semantic_chunk_lengths.append(len(chunk.page_content))
print(semantic_chunk_lengths[:50])
print(len(semantic_chunks))

[209, 418, 534, 476, 752, 1815, 543, 1387, 522, 1979, 2254, 364, 1932, 1338, 1122, 2040, 1780, 750, 1815, 932, 3264, 20, 1199, 1767, 2152, 958, 1386, 1941, 1683, 10, 143, 39, 78, 280, 232, 1065, 1189, 1286, 1102, 134, 2390, 233, 463, 2232, 1877, 897, 173, 1780, 488, 87]
1468


- We observe that the number of chunk lengths is almost as much as fixed chunking
- Some chunks are extremely long, reaching 2152 characters, while some chunks are really small, like 10 characters, hence there is sort of irregularity

## 3. Embedding and Storing

In [12]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [13]:
my_persist_directory="db/chroma_db"
vector_store=Chroma.from_documents(documents=semantic_chunks,
                                   embedding=embeddings,
                                   persist_directory=my_persist_directory,
                                  )
print("Vector Store Created")


Vector Store Created


In [14]:
question="What does the UDA manifesto say?"
matches=vector_store.similarity_search(question, k=3)

for match in matches:
    print(match.page_content[:180])


Two reasons. First, to earn a place at the table, one must first 
emerge as the undisputed kingpin of his com-
munity, by finishing all other contenders for the 
throne.Second it i
Two reasons. First, to earn a place at the table, one must first 
emerge as the undisputed kingpin of his com-
munity, by finishing all other contenders for the 
throne.Second it i
3
T
he election ahead of us is a pivotal one. We 
have three issues on the ballot and how 
we chose will determine whether Kenya 
marches forward or backwards. My first agenda is t


## Importing the LLM to add context to it

In [15]:
from langchain_openai import ChatOpenAI

In [17]:
context="\n\n".join(match.page_content for match in matches)
model= ChatOpenAI(model="gpt-4o-mini")
prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer, say so:

Context:
{context}

Question: {question}

"""
answer = model.invoke(prompt)
print(answer.content)

The UDA manifesto emphasizes the ideology of political inclusion and economic empowerment for all Kenyans. It asserts that the national aspirations of justice, peace, freedom, and prosperity bind citizens to a covenant that allows for participation in decision-making and accountability of leaders. The party rejects the need for patrons and political gatekeepers and advocates for national democratic political parties that enable equality in leadership opportunities, regardless of tribe, gender, physical disability, or social background. Additionally, the manifesto highlights the urgent issue of job creation, noting that many young Kenyans struggle to find stable employment.


# Making the RAG History Aware

 - We want to store previous questions and answers, enabling a chat bot specific kind of interaction

In [18]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [30]:

chat_history = []

def ask_question(user_question):
    if chat_history:
        messages = [SystemMessage(content=(
            "Given the chat history, rewrite the new question to be standalone."
            "Return only the rewritten question, nothing else"
        ))
        ] + chat_history + [
            HumanMessage(content=f"New Question: {user_question}")
        ]
        # Converting the newly generated search question into a vector
        search_question = model.invoke(messages).content.strip()
    else:
        search_question = user_question

    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    # Executing the similarity search
    docs= retriever.invoke(search_question)

    context = "\n".join(f"- {doc.page_content}" for doc in docs)

    combined_input = f"""
    Based on the following documents, please answer this question: {user_question}
    Documents:

    {context}
    Provide a clear, helpful answer using only the information from these documents.
    If the documents do not contain the answer, say so instead of guessing.
    """
    messages = [
        SystemMessage(content="You are a helpful assistant"),
        HumanMessage(content=combined_input)
    ]

    answer = model.invoke(messages).content

    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))

    return answer

In [34]:
def start_chat():
    print("Ask me questions! Type 'quit' to exit. ")
    while True:
        question = input("\n Question:")

        if question.lower() == "quit":
            break
        answer=ask_question(question)
        print()
        print(answer)

In [35]:
start_chat()

Ask me questions! Type 'quit' to exit. 



 Question: What does the constitution state on Murder



The Constitution of Kenya, 2010 addresses the issue of murder implicitly through its provision on the right to life. According to Article 26:

1. Every person has the right to life (Section 26(1)).
2. A person shall not be deprived of life intentionally, except to the extent authorized by the Constitution or other written law (Section 26(3)).

This indicates that murder, defined as the intentional deprivation of life, is contrary to the right to life guaranteed by the Constitution. The Constitution allows for deprivation of life only under specific conditions permitted by law, reinforcing the prohibition of murder. 

Therefore, while the Constitution does not explicitly define murder, it states that intentionally taking a person's life is generally not permissible unless specifically authorized by law.



 Question: What is it



The documents outline several key rights and fundamental freedoms as stipulated in the Constitution of Kenya, 2010. They specifically mention the following:

1. **Right to Life**: 
   - Every person has the right to life, which begins at conception.
   - A person shall not be deprived of life intentionally, except as authorized by the Constitution or other written law.
   - Abortion is not permitted unless there is a need for emergency treatment, or the life or health of the mother is in danger, or if allowed by other written law.

2. **Equality and Freedom from Discrimination**: 
   - The documents highlight the right to equality and freedom from discrimination.

3. **Additional Rights**:
   - Freedom from slavery or servitude.
   - The right to a fair trial.
   - The right to an order of habeas corpus.

This summary encapsulates the rights and freedoms as detailed in the provided excerpts from the Constitution of Kenya, 2010.



 Question: What does the IEBC voter regulation document say on things that may be election irregularities



The IEBC voter regulation document, as reflected in the Election Offences Act (Cap. 66) of Kenya, outlines several actions that may constitute election irregularities. These include:

1. **Unauthorized Voter Registration**:
   - Making, preparing, printing, or possessing documents that purport to be a register of voters without authorization.
   - Making false statements in connection with applications for voter registration.
   - Aiding or abetting the commission of the above offences.

2. **Multiple Registrations**:
   - Applying to be registered as a voter while already registered in another register unless it is a substitution.
   - Applying for registration in the same register more than once, or while an application is still pending (not yet granted or rejected).
   - Making simultaneous applications for voter registration.
   - Applying for voter registration after being disqualified by an election court.

3. **Possession and Use of Ballot Papers**:
   - Possessing marked ballo


 Question: quit
